<a href="https://colab.research.google.com/github/Jaswanth431/DL-Assignment-1/blob/main/DL_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
# @title
!pip install wandb

In [59]:
#importing packages
from keras.datasets import fashion_mnist
import pandas as pd
import numpy as np
import wandb
import math


In [60]:
#creating wandb connection
wandb.login(key="62cfafb7157dfba7fdd6132ac9d757ccd913aaaf")
print("Wandb connection initiated")


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Wandb connection initiated


In [61]:
#getting training and test data
[(x_total_train_data, y_total_train_data), (x_test_data, y_test_data)] = fashion_mnist.load_data()
total_train_len = len(x_total_train_data);
train_count = int(total_train_len*.90)
flattened_train_data = []
#flattening the 28*28 pixel matrix
for i in range(0, total_train_len):
    flattened_train_data.append(x_total_train_data[i].flatten())
    # print(flattened_train_data[i])

x_train_data = np.array(flattened_train_data[0:train_count])
y_train_data = y_total_train_data[0:train_count]
x_validation_data = np.array(flattened_train_data[train_count:])
y_validation_data = y_total_train_data[train_count:]
# print(y_train_data[:100])
# print(x_train_data[0])


In [62]:
#Creating neural network
class NeuralNetwork:
    def __init__(self, input_layer_neurons, output_layer_neurons, config):
        #initializing values
        self.hidden_layers = config.hidden_layers
        self.hidden_layer_neurons = config.hl_size
        self.input_layer_neurons = input_layer_neurons
        self.output_layer_neurons = output_layer_neurons
        self.total_layers = self.hidden_layers+1
        self.output_layer_number = self.total_layers - 1;
        self.config = config

        #input weight and bias initilization
        self.w = []
        self.b = []

        #weights and bias initialization for hidden layers
        for i in range(0, self.total_layers):
          if i == 0:
            temp1 = np.random.randn(self.hidden_layer_neurons, self.input_layer_neurons)
            temp2 = np.random.randn(self.hidden_layer_neurons,1)
          elif i==self.total_layers -1:
            temp1 = np.random.randn(self.output_layer_neurons, self.hidden_layer_neurons)
            temp2 = np.random.randn(self.output_layer_neurons,1)
          else:
            temp1  =  np.random.randn( self.hidden_layer_neurons, self.hidden_layer_neurons)
            temp2 = np.random.randn(self.hidden_layer_neurons,1)

          self.w.append(temp1*0.01)
          self.b.append(temp2*0.01)

        print("Init:",self.w)
        # print(self.b)
        # # print(self.w, self.b)
        # for i in range(0, self.total_layers):
        #    print(i, self.w[i].shape)
        #    print(i, self.b[i].shape)
        # # print(self.w[2].shape)












        # self.update_parameters(d_w, d_b, self.config.learning_rate)

    def update_parameters(self,d_w,d_b, eta):
      for i in range(0, self.total_layers):
        self.w[i] -= eta*d_w[i]
        self.b[i] -= eta*d_b[i]

    def sigmoid(self,arr):
        # print(arr)
        clip_arr = np.clip(arr,-500, 500)
        return 1. / (1.+np.exp(-clip_arr))
    def sigmoid_derivative(self, arr):
        return self.sigmoid(arr) * (1-self.sigmoid(arr))
    def softmax(self, arr):
        return np.exp(arr) / np.sum(np.exp(arr), axis=0)



In [63]:
#forward prop
def forward_propogate(self, input, flag=False):
        h = [None] * self.total_layers
        a = [None] * self.total_layers

        for i in range(0, self.total_layers):
            if(i == 0):
              a[i] = np.matmul(self.w[i],input.reshape(self.input_layer_neurons,1) ) + self.b[i]
              h[i] = self.sigmoid(a[i])

            elif i == self.total_layers-1:
              a[i] = np.matmul(self.w[i],h[i-1] ) + self.b[i]
              h[i] = self.softmax(a[i])
              if(flag):
                print("A:",a[i])
                print("H:",h[i])
            else:
              a[i] = np.matmul(self.w[i],h[i-1] ) + self.b[i]
              h[i] = self.sigmoid(a[i])
            # print(a[i].shape)
        return h, a
NeuralNetwork.forward_propogate  = forward_propogate

In [64]:
#back prop
def back_propagation(self, h, a, actual_class, input_pixels):
       d_h = [None] * self.total_layers
       d_a =  [None] * self.total_layers
       d_w =  [None] * self.total_layers
       d_b =  [None] * self.total_layers
       y_original = np.zeros((self.output_layer_neurons, 1))
       y_original[actual_class] = 1
      #  print(actual_class, y_original)

       #gradient w.r.p.t output y hat
       d_a[self.total_layers-1] = -(y_original - h[self.total_layers-1])
      #  print("a ", self.total_layers-1, d_a[self.total_layers-1])
       for i in range(self.total_layers-1, -1, -1):
        if(i == 0):
          d_w[i] = np.matmul(d_a[i], input_pixels.reshape(1, -1))
          # print("d_a ", i, d_a[i])
          # print("input pixel ", -1,input_pixels.reshape(1, -1))
          # print("d_w", i, d_w[i])

        else:
          d_w[i] = np.matmul(d_a[i], h[i-1].T)
          # print("d_a ", i, d_a[i])
          # print("h ", i-1, h[i-1].T)
          # print("d_w", i, d_w[i])

        d_b[i] = np.copy(d_a[i])

        if(i-1>=0):
          d_h[i-1]=np.matmul(self.w[i].T,d_a[i])
          # print("w ", i, self.w[i])
          # print("d_a ", i, d_a[i])
          # print("d_h", i-1, d_h[i-1])

          d_a[i-1] = d_h[i-1] * self.sigmoid_derivative(a[i-1])
          # print("d_h ", i-1, d_h[i-1])
          # print("a at", i-1, a[i-1])
          # print("sig der at a", i-1, self.sigmoid_derivative(a[i-1]))
          # print("d_a ", i-1, d_a[i-1])


      #  print(d_w)
      #  print(d_b)

       return d_w, d_b
NeuralNetwork.back_propagation  = back_propagation

In [65]:
#stochastic gd
def stochastic_gradient_descent(self, x_train_data, y_train_data):
      for i in range(0, self.config.epochs):
        # d_w = [np.zeros_like(weight) for weight in self.w]
        # d_b = [np.zeros_like(bias) for bias in self.b]
        for j in range(0, len(x_train_data)):
          h,a = self.forward_propogate(x_train_data[j])
          d_w_temp, d_b_temp = self.back_propagation(h, a, y_train_data[j], x_train_data[j])
          for k in range(self.total_layers):
                self.w[k] -= self.config.learning_rate*d_w_temp[k]
                self.b[k] -= self.config.learning_rate*d_b_temp[k]
NeuralNetwork.stochastic_gradient_descent  = stochastic_gradient_descent

In [66]:
#momentum gd
def momentum_gradient_descent(self, x_train_data, y_train_data):
      previous_w = [np.zeros_like(weight) for weight in self.w]
      previous_b = [np.zeros_like(bias) for bias in self.b]
      temp_w = [np.zeros_like(weight) for weight in self.w]
      temp_b = [np.zeros_like(bias) for bias in self.b]
      beta =0.9
      for i in range(0, self.config.epochs):
        d_w = [np.zeros_like(weight) for weight in self.w]
        d_b = [np.zeros_like(bias) for bias in self.b]


        for j in range(0, len(x_train_data)):
          h,a = self.forward_propogate(x_train_data[j])
          d_w_temp, d_b_temp = self.back_propagation(h, a, y_train_data[j], x_train_data[j])
          # print("A:",d_w_temp[0])
          # print("B:",d_b_temp[0])
          # print("C:",d_b_temp[1])
          # print("D:",d_b_temp[2])

          for k in range(self.total_layers):
                d_w[k] += d_w_temp[k]
                d_b[k] += d_b_temp[k]
        for k in range(self.total_layers):
                temp_w[k] = beta * previous_w[k] + self.config.learning_rate*d_w[k]
                self.w[k] -= temp_w[k]
                previous_w[k] = temp_w[k]
                temp_b[k] = beta * previous_b[k] + self.config.learning_rate*d_b[k]
                self.b[k] -= temp_b[k]
                previous_b[k] = temp_b[k]
        # print(self.w[0])
        # print(self.b)
NeuralNetwork.momentum_gradient_descent = momentum_gradient_descent

In [67]:
#Standard gradient descent
def standard_gradient_descent(self, x_train_data, y_train_data):
      for i in range(0, self.config.epochs):
        d_w = [np.zeros_like(weight) for weight in self.w]
        d_b = [np.zeros_like(bias) for bias in self.b]
        for j in range(0, len(x_train_data)):
          h,a = self.forward_propogate(x_train_data[j])
          d_w_temp, d_b_temp = self.back_propagation(h, a, y_train_data[j], x_train_data[j])
          for k in range(self.total_layers):
                d_w[k] += d_w_temp[k]
                d_b[k] += d_b_temp[k]

        self.update_parameters(d_w, d_b, self.config.learning_rate)
        # print(self.w[0])
NeuralNetwork.standard_gradient_descent = standard_gradient_descent

In [68]:
#nestro gradient
def nestro_gradient_descent(self, x_train_data, y_train_data):
      previous_w = [np.zeros_like(weight) for weight in self.w]
      previous_b = [np.zeros_like(bias) for bias in self.b]
      temp_w = [np.zeros_like(weight) for weight in self.w]
      temp_b = [np.zeros_like(bias) for bias in self.b]
      beta =0.9
      for i in range(0, self.config.epochs):
        d_w = [np.zeros_like(weight) for weight in self.w]
        d_b = [np.zeros_like(bias) for bias in self.b]

        for k in range(self.total_layers):
          temp_w[k] = beta*previous_w[k]
          temp_b[k] = beta * previous_b[k]


        self.update_parameters(previous_w, previous_b, beta)

        for j in range(0, len(x_train_data)):
          h,a = self.forward_propogate(x_train_data[j])
          d_w_temp, d_b_temp = self.back_propagation(h, a, y_train_data[j], x_train_data[j])
          # print("A:",d_w_temp[0])
          # print("B:",d_b_temp[0])
          # print("C:",d_b_temp[1])
          # print("D:",d_b_temp[2])

          for k in range(self.total_layers):
                d_w[k] += d_w_temp[k]
                d_b[k] += d_b_temp[k]
        for k in range(self.total_layers):
                previous_w[k] = temp_w[k] + self.config.learning_rate*d_w[k]
                self.w[k] -= self.config.learning_rate*d_w[k]
                previous_b[k] = temp_b[k] + self.config.learning_rate*d_b[k]
                self.b[k] -= self.config.learning_rate*d_b[k]
        # print(self.w[0])
        # print(self.b)
NeuralNetwork.nestro_gradient_descent = nestro_gradient_descent

In [69]:
#loss calculation
def calculate_loss(self, x_train_data, y_train_data, x_validation_data, y_validation_data):
      train_count = 0
      train_loss = 0
      validation_loss = 0
      validation_count = 0
      for i in range(len(x_train_data)):
        h,a = self.forward_propogate(x_train_data[i])
        output_class = np.argmax(h[self.total_layers-1])
        # print("A", x_train_data[i])
        # print("B", h[self.total_layers-1])
        # print("C", output_class)
        # print("D", y_train_data[i])

        actual_class = y_train_data[i]
        if(output_class == actual_class):
          train_count+=1
        train_loss+=-math.log10(h[self.total_layers-1][actual_class])

      for i in range(len(x_validation_data)):
        h,a = self.forward_propogate(x_validation_data[i])
        output_class = np.argmax(h[self.total_layers-1])
        # print("A", x_validation_data[i])
        # print("B", h[self.total_layers-1])
        # print("C", output_class)
        # print("D", y_train_data[i])

        actual_class = y_validation_data[i]
        if(output_class == actual_class):
          validation_count+=1
        validation_loss+=-math.log10(h[self.total_layers-1][actual_class])
      print(train_count, validation_count)
      print(train_loss, validation_loss)
      train_accuracy = train_count/len(x_train_data)
      validation_accuracy = validation_count/len(x_validation_data)
      train_loss = train_loss(len(x_train_data))
      validation_loss = validation_loss/(len(x_validation_data))
      print(train_accuracy,train_loss, validation_accuray, validation_loss)
      wandb.log({"train_accuracy":train_accuracy, "train_loss":train_loss, "val_accuracy":validation_accuracy, "val_loss":validation_loss})
NeuralNetwork.calculate_loss  = calculate_loss

In [70]:
def gradient_descent(x_train_data, y_train_data):
  if(self.config.optimizer == "sgd"):
    self.stochastic_gradient_descent(x_train_data, y_train_data)
    self.calculate_loss(x_train_data, y_train_data, x_validation_data, y_validation_data)


NeuralNetwork.gradient_descent = gradient_descent

In [71]:
# n_network.standard_gradient_descent(x_train_data, y_train_data)
# n_network.stochastic_gradient_descent(x_train_data, y_train_data)
# n_network.stochastic_gradient_descent(x_train_data, y_train_data)
# n_network.momentum_gradient_descent(x_train_data, y_train_data)
# n_network.nestro_gradient_descent(x_train_data, y_train_data)


# n_network.calculate_train_loss(x_train_data, y_train_data)
# h, a = n_network.forward_propogate(x_train_data[0])
# d_w, d_b = n_network.back_propagation(h, a, y_train_data[0], x_train_data[0])

In [72]:
# h1,a1 = n_network.forward_propogate(x_train_data[0], True)


In [73]:
# h1,a1 = n_network.forward_propogate(x_train_data[1], True)
# h1,a1 = n_network.forward_propogate(x_train_data[2], True)
# h1,a1 = n_network.forward_propogate(x_train_data[3], True)

# d_w, d_b = n_network.back_propagation(h1, a1, y_train_data[0], x_train_data[0])
# print(h1)
# print(a1)
# h2,a2 = n_network.forward_propogate(x_train_data[1])
# print(h2)
# print(a2)

In [74]:
h_param_config = {
    "epochs":5,
    "hidden_layers":3,
    "hl_size":32,
    "weight_decay":0.5,
    "learning_rate":0.0001,
    "optimizer":"sgd",
    "batch_size":16,
    "initialization":"random",
    "activation":"sigmoid",
    "loss_type":"cross_entropy"
}


In [75]:
def train(config):
  print(config)
  run = wandb.init(project="DL assignment 1", name = f"{config['optimizer']}_hl_{config['hidden_layers']}_hls_{config['hl_size']}_bs_{config['batch_size']}_ac_{config['activation']}", config=config)
  print(wandb.config)
  n_network = NeuralNetwork(784, 10, config)
  n_network.gradient_descent(x_train_data, y_train_data)
  n_network.calculate_loss(x_train_data, y_train_data, x_validation_data, y_validation_data)

train(h_param_config)


{'epochs': 5, 'hidden_layers': 3, 'hl_size': 32, 'weight_decay': 0.5, 'learning_rate': 0.0001, 'optimizer': 'sgd', 'batch_size': 16, 'initialization': 'random', 'activation': 'sigmoid', 'loss_type': 'cross_entropy'}


accuracy,▁▁
loss,▁▁
accuracy,0.10844
loss,0.99855


{'epochs': 5, 'hidden_layers': 3, 'hl_size': 32, 'weight_decay': 0.5, 'learning_rate': 0.0001, 'optimizer': 'sgd', 'batch_size': 16, 'initialization': 'random', 'activation': 'sigmoid', 'loss_type': 'cross_entropy'}


AttributeError: 'dict' object has no attribute 'hidden_layers'

In [ ]:
# n_network = NeuralNetwork(1, 10 , 784, 10, wandb.config)

In [ ]:
n_network.calculate_loss(x_train_data, y_train_data,x_validation_data, y_validation_data )
